# Del modelo entrenado al modelo consumido por API

**Objetivo de aprendizaje:**
Al finalizar este notebook, el alumno podrá:
- Entender la transición histórica de entrenar modelos propios a consumir APIs
- Comparar costes, riesgos y velocidad de desarrollo de los tres enfoques
- Identificar cuando es un error estratégico entrenar un modelo propio
- Aplicar el árbol de decisión make vs buy a casos reales de la empresa

**Conexión con el documento:**
Este notebook acompaña la sección `### 5.9 Del modelo entrenado al modelo
consumido por API`. Es un notebook conceptual: las figuras son visualizaciones
de datos estratégicos, no demostraciones de ML.

> **Antes de seguir:** si la empresa necesita clasificar automáticamente
> los tickets de soporte por urgencia, ¿entrenarias un modelo propio
> o usarias una API? Anota tu respuesta y el razonamiento.
> Al final del notebook tendras los criterios para revisarla.

<details>
<summary>Por que esta pregunta importa (desplegar)</summary>

La mayoria de los proyectos de IA en empresa que fracasan no fallan
por problemas técnicos. Fallan porque eligieron el enfoque equivocado
antes de escribir una sola linea de codigo.
Entrenar cuando deberias usar API desperdicia meses.
Usar API cuando deberias entrenar crea dependencias que luego son
inaceptables para el cliente o para el regulador.

</details>

## Setup

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

IMG = 'images'
os.makedirs(IMG, exist_ok=True)
print('Setup OK')

Setup OK


## 1. Las tres épocas: de entrenar a consumir

La transición no fue gradual: fue una serie de discontinuidades.
Cada hito cambió radicalmente lo que era viable para una empresa
sin un laboratorio de investigación propio.

**Lo que cambia en cada época no es la técnica: es el coste de acceso.**

In [2]:
# Linea de tiempo: hitos que cambiaron el acceso a IA
hitos = [
    (2012, 'AlexNet\n(ImageNet)', 1, '#1565C0'),
    (2014, 'GAN\n(Goodfellow)', 1, '#1565C0'),
    (2017, 'Transformer\n(Attention is All)', 1, '#1565C0'),
    (2018, 'BERT\n(Google, open)', 2, '#2E7D32'),
    (2019, 'GPT-2\n(OpenAI, open)', 2, '#2E7D32'),
    (2020, 'GPT-3\n(API privada)', 2, '#2E7D32'),
    (2022, 'ChatGPT\n(acceso masivo)', 3, '#B71C1C'),
    (2023, 'GPT-4 API\nClaude API', 3, '#B71C1C'),
    (2024, 'Modelos open\nLlama3, Mistral', 3, '#B71C1C'),
]

fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(2010, 2026)
ax.set_ylim(0, 4.5)
ax.axis('off')

# Linea base
ax.axhline(1.5, color='#555', lw=2, zorder=1)

# Bandas de epoca
epocas = [
    (2012, 2019, '#E3F2FD', 'Epoca 1\nModelos propios\n(solo grandes labs)'),
    (2019, 2022, '#E8F5E9', 'Epoca 2\nTransfer learning\n(accesible con GPU)'),
    (2022, 2025, '#FFEBEE', 'Epoca 3\nAPI-first\n(cualquier empresa)'),
]
for x0, x1, color, label in epocas:
    ax.axvspan(x0, x1, ymin=0, ymax=1, alpha=0.3, color=color)
    ax.text((x0+x1)/2, 0.15, label, ha='center', va='bottom',
            fontsize=9, color='#333')

# Hitos
for (year, nombre, nivel, color) in hitos:
    y_hito = 1.5 + (0.6 if nivel == 1 else 1.1 if nivel == 2 else 1.6)
    ax.plot([year, year], [1.5, y_hito], color=color, lw=1.5, zorder=2)
    ax.scatter(year, y_hito, color=color, s=60, zorder=3)
    ax.text(year, y_hito + 0.1, nombre, ha='center', va='bottom',
            fontsize=8, color=color, fontweight='bold')

# Eje de anio
for yr in range(2012, 2026, 2):
    ax.text(yr, 1.3, str(yr), ha='center', va='top', fontsize=9, color='#555')

ax.set_title('Del laboratorio de investigacion a la llamada HTTP: '
             'tres epocas de acceso a IA', fontsize=13, pad=10)
plt.tight_layout()
plt.savefig('images/B05E_fig01.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B05E_fig01.png')

Guardado: B05E_fig01.png


## 2. Comparativa de costes: órdenes de magnitud

La diferencia de coste entre los tres enfoques no es lineal: es logarítmica.
Entrenar un modelo del tamaño de GPT-3 cuesta entre 4 y 12 millones de dólares
en cómputo. Fine-tuning de un modelo base: entre 1.000 y 50.000 dólares.
Una llamada a API de Claude o GPT-4: entre 0.003 y 0.06 dólares por 1.000 tokens.

**Para un caso típico de empresa software** (clasificar 10.000 documentos al mes):
- Modelo propio desde cero: inviable sin equipo dedicado
- Fine-tuning: ~2.000 euros/mes (GPU + datos + ingeniería)
- API: ~15-80 euros/mes según modelo elegido

La ventaja de la API no es solo el coste directo:
es el coste de oportunidad de no dedicar el equipo a ML interno.

In [3]:
# Comparativa de costes: escala logaritmica
enfoques = ['Entrenar desde cero\n(GPT-3 scale)', 'Fine-tuning\n(modelo base)',
            'API directa\n(GPT-4 / Claude)']
costes_min = [2_000_000, 500, 15]
costes_max = [12_000_000, 50_000, 80]
costes_mid = [(a+b)/2 for a, b in zip(costes_min, costes_max)]

colores = ['#E53935', '#FB8C00', '#43A047']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Barras log de coste
ax = axes[0]
bars = ax.barh(enfoques, costes_mid, color=colores, alpha=0.8,
               xerr=[costes_mid, costes_mid], ecolor='gray',
               capsize=4)
ax.set_xscale('log')
ax.set_xlabel('Coste estimado mensual (euros, escala logaritmica)', fontsize=11)
ax.set_title('Coste operativo mensual por enfoque\n'
             'caso: 10.000 documentos/mes', fontsize=12)
for bar, mid, mn, mx in zip(bars, costes_mid, costes_min, costes_max):
    ax.text(mx * 1.5, bar.get_y() + bar.get_height()/2,
            f'{mn:,.0f} - {mx:,.0f} eur',
            va='center', fontsize=9)

# Radar de dimensiones estrategicas
ax2 = axes[1]
dimensiones = ['Coste\ninicial', 'Velocidad\nal mercado', 'Control\ndel modelo',
               'Privacidad\ndel dato', 'Escalabilidad']
# Puntuacion 1-5 (5=mejor para la empresa)
scores = {
    'Modelo propio':  [1, 1, 5, 5, 2],
    'Fine-tuning':    [3, 3, 4, 5, 3],
    'API directa':    [5, 5, 1, 2, 5],
}
x = np.arange(len(dimensiones))
width = 0.25
for i, (nombre, vals) in enumerate(scores.items()):
    ax2.bar(x + i*width, vals, width, label=nombre,
            color=colores[i], alpha=0.8)
ax2.set_xticks(x + width)
ax2.set_xticklabels(dimensiones, fontsize=9)
ax2.set_ylabel('Puntuacion (5 = mas favorable)', fontsize=11)
ax2.set_title('Perfil estrategico por enfoque\n'
              '(perspectiva empresa software)', fontsize=12)
ax2.legend(fontsize=9)
ax2.set_ylim(0, 6)
ax2.axhline(3, color='gray', lw=0.8, linestyle='--', alpha=0.5)

plt.suptitle('Comparativa estrategica: tres enfoques de integracion de IA',
             fontsize=13)
plt.tight_layout()
plt.savefig('images/B05E_fig02.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B05E_fig02.png')

Guardado: B05E_fig02.png


## 3. El moat real: datos propietarios vs modelo

Cuando dos empresas usan la misma API, el modelo no es la ventaja.
La ventaja esta en lo que cada empresa aporta al contexto del modelo.

**Tres tipos de ventaja posible en un mundo API-first:**

1. **Dato propietario**: historial de clientes, contratos, comportamiento
   de uso del producto. Nadie mas tiene ese dato. Se puede usar en RAG
   o fine-tuning para producir un sistema que mejora con el tiempo.

2. **Proceso propietario**: la decisión de que automatizar y como integrarlo
   en el flujo de trabajo. Una empresa que integra bien IA en su ciclo
   de ventas tiene ventaja operativa aunque use la misma API que la competencia.

3. **Velocidad de iteración**: quien aprende mas rapido a mejorar sus prompts,
   su arquitectura RAG y su pipeline de evaluación, acumula ventaja compuesta
   que los demas no pueden replicar de un dia para otro.

**Lo que NO es ventaja competitiva sostenible:**
- Ser los primeros en usar ChatGPT para una tarea
- Tener acceso a GPT-4 (todos lo tienen)
- Haber construido un wrapper sobre una API

In [4]:
# Diagrama: fuentes de ventaja en empresa API-first
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(0, 12)
ax.set_ylim(0, 7)
ax.axis('off')

def caja(ax, x, y, w, h, color, texto, subtexto='', borde='#333'):
    rect = FancyBboxPatch((x, y), w, h,
                          boxstyle='round,pad=0.15',
                          linewidth=1.5, edgecolor=borde,
                          facecolor=color)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2 + (0.15 if subtexto else 0),
            texto, ha='center', va='center',
            fontsize=10, fontweight='bold', color=borde)
    if subtexto:
        ax.text(x + w/2, y + h/2 - 0.25, subtexto,
                ha='center', va='center', fontsize=8, color='#555')

def flecha(ax, x0, y0, x1, y1):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

# Modelo fundacional (base comun)
caja(ax, 4.5, 0.3, 3, 0.9, '#E3F2FD', 'Modelo Fundacional (API)',
     'Igual para todos', '#0D47A1')

# Empresa A (con ventaja)
caja(ax, 0.5, 2.5, 4.5, 1.0, '#E8F5E9', 'Empresa A',
     'Dato propio + RAG + proceso integrado', '#1B5E20')
# Empresa B (sin ventaja)
caja(ax, 7.0, 2.5, 4.5, 1.0, '#FFEBEE', 'Empresa B',
     'Solo wrapper sobre API', '#B71C1C')

# Resultado
caja(ax, 0.5, 5.0, 4.5, 1.0, '#C8E6C9', 'Ventaja compuesta',
     'Mejora con cada interaccion', '#1B5E20')
caja(ax, 7.0, 5.0, 4.5, 1.0, '#FFCDD2', 'Commodity',
     'Replicable en horas por cualquier competidor', '#B71C1C')

# Flechas
flecha(ax, 6.0, 1.2, 2.75, 2.5)   # API -> Empresa A
flecha(ax, 6.0, 1.2, 9.25, 2.5)   # API -> Empresa B
flecha(ax, 2.75, 3.5, 2.75, 5.0)  # A -> ventaja
flecha(ax, 9.25, 3.5, 9.25, 5.0)  # B -> commodity

# Capa de datos propios (solo empresa A)
caja(ax, 0.5, 4.0, 4.5, 0.8, '#DCEDC8', 'Datos propios + contexto',
     'CRM, contratos, historial de uso', '#33691E')
flecha(ax, 2.75, 4.0, 2.75, 3.5)

ax.set_title('El modelo es igual para todos. La ventaja esta en el contexto.',
             fontsize=13, pad=15)
plt.tight_layout()
plt.savefig('images/B05E_fig03.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B05E_fig03.png')

Guardado: B05E_fig03.png


## 4. Cuando entrenar es obligatorio (y cuando es un error)

**Casos donde NO se puede usar API externa:**

| Caso | Razón | Alternativa |
|---|---|---|
| Dato médico o financiero regulado | GDPR, HIPAA, normativa sectorial: el dato no puede salir | Modelo propio en infraestructura privada |
| Contrato de confidencialidad estricto | El cliente prohibe explicitamente procesar su dato en terceros | On-premise o modelo local |
| Latencia < 50ms requerida | Una llamada HTTP típica: 200-800ms | Modelo local cuantizado (llama.cpp, ONNX) |
| Volumen masivo con coste de API prohibitivo | > 100M tokens/mes: el coste de API supera el de operar modelo propio | Fine-tuning de modelo pequeño especializado |

**Casos donde entrenar es un error:**

| Caso | Por que es un error |
|---|---|
| La tarea ya la resuelve un modelo fundacional con buen prompting | Meses de trabajo para llegar al mismo resultado |
| No existe equipo de ML en la empresa | El modelo se degrada sin mantenimiento continuo |
| El go-to-market es en semanas | Preparar datos + entrenar + validar: mínimo 2-3 meses |
| El problema cambia frecuentemente | Reentrenar es mas caro que actualizar el sistema de prompts |

In [5]:
# Arbol de decision: make vs buy para modelos de IA
fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')

def nodo(ax, x, y, texto, color='#E3F2FD', borde='#1565C0',
         w=2.8, h=0.7, fs=9):
    rect = FancyBboxPatch((x - w/2, y - h/2), w, h,
                          boxstyle='round,pad=0.1',
                          linewidth=1.5, edgecolor=borde, facecolor=color)
    ax.add_patch(rect)
    ax.text(x, y, texto, ha='center', va='center',
            fontsize=fs, color=borde, fontweight='bold',
            wrap=True)

def flecha_et(ax, x0, y0, x1, y1, etiqueta='', lado='left'):
    ax.annotate('', xy=(x1, y1+0.35), xytext=(x0, y0-0.35),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.2))
    mx, my = (x0+x1)/2, (y0+y1)/2
    ha = 'right' if lado == 'left' else 'left'
    ax.text(mx + (-0.15 if lado == 'left' else 0.15), my,
            etiqueta, ha=ha, va='center', fontsize=8, color='#555',
            style='italic')

# Raiz
nodo(ax, 7, 9.3, '¿El dato puede salir del perimetro?', '#FFF9C4', '#F57F17', w=4)

# Rama NO
nodo(ax, 2.5, 7.5, 'Modelo propio\no fine-tuning en infra propia',
     '#FFCDD2', '#B71C1C', w=3.5)
flecha_et(ax, 7, 9.3, 2.5, 7.5, 'No', 'left')

# Rama SI -> pregunta 2
nodo(ax, 10, 7.5, '¿Modelo fundacional resuelve\nla tarea con buen prompting?',
     '#FFF9C4', '#F57F17', w=4)
flecha_et(ax, 7, 9.3, 10, 7.5, 'Si', 'right')

# Rama SI -> pregunta 3 (volumen)
nodo(ax, 8, 5.8, '¿El volumen hace rentable\nel coste de API?',
     '#FFF9C4', '#F57F17', w=3.5)
flecha_et(ax, 10, 7.5, 8, 5.8, 'Si', 'left')

# API directa
nodo(ax, 6, 4.1, 'API + RAG', '#C8E6C9', '#1B5E20', w=2.5)
flecha_et(ax, 8, 5.8, 6, 4.1, 'Si', 'left')

# Fine-tuning modelo pequeno
nodo(ax, 10.5, 4.1, 'Fine-tuning modelo\npequeno especializado',
     '#FFE0B2', '#E65100', w=3)
flecha_et(ax, 8, 5.8, 10.5, 4.1, 'No', 'right')

# Rama NO -> dato suficiente?
nodo(ax, 12, 5.8, '¿Hay dato propietario\nsuficiente para entrenar?',
     '#FFF9C4', '#F57F17', w=3.5)
flecha_et(ax, 10, 7.5, 12, 5.8, 'No', 'right')

# Fine-tuning propio
nodo(ax, 11, 4.1, 'Fine-tuning o modelo propio', '#FFE0B2', '#E65100', w=3)
flecha_et(ax, 12, 5.8, 11, 4.1, 'Si', 'left')

# Redefinir problema
nodo(ax, 13.5, 4.1, 'Redefinir problema\no recolectar datos',
     '#FFCDD2', '#B71C1C', w=2.8)
flecha_et(ax, 12, 5.8, 13.5, 4.1, 'No', 'right')

ax.set_title('Arbol de decision: make vs buy para modelos de IA en empresa software',
             fontsize=13, pad=10)
plt.tight_layout()
plt.savefig('images/B05E_fig04.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B05E_fig04.png')

Guardado: B05E_fig04.png


## 5. Ejercicio técnico: calculadora de TCO (Total Cost of Ownership)

**Caso la empresa:**
El equipo evalua tres opciones para implementar clasificación automática
de contratos de clientes (urgente / revisión / estándar).

- Volumen estimado: 500 contratos/mes
- Longitud media: 2.000 tokens por contrato
- Tiempo de vida del proyecto: 12 meses

Calcula el TCO a 12 meses de cada enfoque.
Incluye: coste de cómputo, coste de ingeniería (dias-persona),
coste de mantenimiento.

In [6]:
# Calculadora de TCO - parametros configurables
params = {
    'contratos_mes': 500,
    'tokens_por_contrato': 2000,
    'meses': 12,
    'coste_dia_ingeniero': 400,  # euros/dia
}

total_tokens_mes = params['contratos_mes'] * params['tokens_por_contrato']
total_tokens_anio = total_tokens_mes * params['meses']

# Escenario A: API directa (GPT-4o mini)
precio_api_por_1M_tokens = 0.15  # input, euros/1M tokens
coste_api_mes = total_tokens_mes / 1_000_000 * precio_api_por_1M_tokens
dias_integracion_api = 5  # setup + prompting + evaluacion
tco_api = (
    coste_api_mes * params['meses']
    + dias_integracion_api * params['coste_dia_ingeniero']
    + 2 * params['coste_dia_ingeniero']  # mantenimiento anual
)

# Escenario B: Fine-tuning modelo pequeno (Mistral 7B)
coste_gpu_finetuning = 150  # euros (4h en A100)
dias_preparacion_datos = 10
dias_finetuning_eval = 8
coste_hosting_mes = 80  # GPU inference
tco_ft = (
    coste_gpu_finetuning
    + (dias_preparacion_datos + dias_finetuning_eval) * params['coste_dia_ingeniero']
    + coste_hosting_mes * params['meses']
    + 5 * params['coste_dia_ingeniero']  # reentrenamiento cada 6 meses
)

# Escenario C: Modelo propio desde cero
dias_desarrollo = 90  # 3 meses de equipo ML
coste_infra_entrenamiento = 3000  # euros
coste_hosting_propio_mes = 200
dias_mantenimiento_mes = 2
tco_propio = (
    dias_desarrollo * params['coste_dia_ingeniero']
    + coste_infra_entrenamiento
    + coste_hosting_propio_mes * params['meses']
    + dias_mantenimiento_mes * params['meses'] * params['coste_dia_ingeniero']
)

print('=== TCO a 12 meses - clasificacion de contratos ===')
print(f'Volumen: {params["contratos_mes"]} contratos/mes x '
      f'{params["tokens_por_contrato"]:,} tokens = '
      f'{total_tokens_mes/1000:.0f}K tokens/mes')
print()
print(f'A) API directa (GPT-4o mini):     {tco_api:,.0f} eur')
print(f'   Coste API/mes:                  {coste_api_mes:.2f} eur')
print(f'   Dias ingeniero: {dias_integracion_api}')
print()
print(f'B) Fine-tuning (Mistral 7B):       {tco_ft:,.0f} eur')
print(f'   GPU hosting/mes:                {coste_hosting_mes} eur')
print(f'   Dias preparacion + eval: {dias_preparacion_datos + dias_finetuning_eval}')
print()
print(f'C) Modelo propio desde cero:       {tco_propio:,.0f} eur')
print(f'   Dias desarrollo: {dias_desarrollo}')
print()
print(f'Ratio C/A: {tco_propio/tco_api:.1f}x mas caro que API')
print(f'Razon de elegir C sobre A: solo si regulacion, privacidad o'
      f' volumen lo justifican')

=== TCO a 12 meses -  clasificacion de contratos ===
Volumen: 500 contratos/mes x 2,000 tokens = 1000K tokens/mes

A) API directa (GPT-4o mini):     2,802 eur
   Coste API/mes:                  0.15 eur
   Dias ingeniero: 5

B) Fine-tuning (Mistral 7B):       10,310 eur
   GPU hosting/mes:                80 eur
   Dias preparacion + eval: 18

C) Modelo propio desde cero:       51,000 eur
   Dias desarrollo: 90

Ratio C/A: 18.2x mas caro que API
Razon de elegir C sobre A: solo si regulacion, privacidad o volumen lo justifican


In [7]:
# Visualizacion TCO comparativo
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Barras de TCO
ax = axes[0]
nombres = ['API directa\n(GPT-4o mini)', 'Fine-tuning\n(Mistral 7B)',
           'Modelo propio\ndesde cero']
valores = [tco_api, tco_ft, tco_propio]
colores_tco = ['#43A047', '#FB8C00', '#E53935']
bars = ax.bar(nombres, valores, color=colores_tco, alpha=0.8)
for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{val:,.0f} eur', ha='center', va='bottom', fontsize=10,
            fontweight='bold')
ax.set_ylabel('TCO 12 meses (euros)', fontsize=11)
ax.set_title('Coste total a 12 meses\n500 contratos/mes, 2K tokens c/u',
             fontsize=12)
ax.set_ylim(0, max(valores) * 1.2)

# Desglose por componente
ax2 = axes[1]
componentes = ['Computo', 'Ingenieria', 'Hosting/Ops']
vals_api = [coste_api_mes * params['meses'],
            dias_integracion_api * params['coste_dia_ingeniero'],
            2 * params['coste_dia_ingeniero']]
vals_ft  = [coste_gpu_finetuning,
            (dias_preparacion_datos + dias_finetuning_eval) * params['coste_dia_ingeniero'],
            coste_hosting_mes * params['meses'] + 5 * params['coste_dia_ingeniero']]
vals_prop = [coste_infra_entrenamiento,
             dias_desarrollo * params['coste_dia_ingeniero'],
             coste_hosting_propio_mes * params['meses']
             + dias_mantenimiento_mes * params['meses'] * params['coste_dia_ingeniero']]

x2 = np.arange(3)
w2 = 0.25
cs = ['#E3F2FD', '#E8F5E9', '#FFEBEE']
bs = ['#1565C0', '#1B5E20', '#B71C1C']
for i, (vlist, col, bcol, lbl) in enumerate([
    (vals_api, cs[0], bs[0], 'API directa'),
    (vals_ft, cs[1], bs[1], 'Fine-tuning'),
    (vals_prop, cs[2], bs[2], 'Modelo propio'),
]):
    bottom = 0
    for j, v in enumerate(vlist):
        ax2.bar(i, v, bottom=bottom, color=col, edgecolor=bcol,
                label=lbl if j == 0 else '')
        if v > 200:
            ax2.text(i, bottom + v/2, componentes[j],
                     ha='center', va='center', fontsize=8, color=bcol)
        bottom += v
ax2.set_xticks([0, 1, 2])
ax2.set_xticklabels(['API directa', 'Fine-tuning', 'Modelo propio'])
ax2.set_ylabel('Euros', fontsize=11)
ax2.set_title('Desglose por componente de coste', fontsize=12)

plt.suptitle('TCO comparativo: la empresa - clasificacion de contratos',
             fontsize=13)
plt.tight_layout()
plt.savefig('images/B05E_fig05.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B05E_fig05.png')

Guardado: B05E_fig05.png


## 6. Ejercicio de decisión

**Caso:** la empresa esta desarrollando un nuevo módulo de IA para análisis
automático de contratos de clientes. El objetivo es identificar clausulas
de penalización, SLAs (Service Level Agreements, acuerdos de nivel de servicio)
y condiciones especiales.

**Contexto:**
- Los contratos contienen información confidencial de clientes
- El equipo tiene 2 desarrolladores .NET con formación básica en IA
- El módulo debe estar en producción en 8 semanas
- Volumen estimado: 200 contratos/mes
- Los contratos son en español, con terminología jurídica específica

**Preguntas:**

1. Usando el árbol de decisión, ¿que enfoque recomendarias?
2. ¿El factor 'confidencialidad' descarta automáticamente el uso de API?
   ¿O depende del acuerdo con el proveedor?
3. Si el volumen crece a 5.000 contratos/mes en 12 meses,
   ¿cambia tu decisión? ¿En que punto?

**Tu razonamiento:**

(escribe aqui tu argumentación)

---

**Criterios de evaluación:**

Una respuesta sólida menciona:
1. La confidencialidad no descarta la API si el proveedor firma un DPA
   (Data Processing Agreement, acuerdo de tratamiento de datos) adecuado.
   Claude Enterprise y GPT-4 Enterprise incluyen este acuerdo.
2. Con 200 contratos/mes y 8 semanas, la API es la única opción viable
   en el tiempo dado. Fine-tuning con terminología jurídica requeriria
   datos etiquetados que no existen aun.
3. A 5.000 contratos/mes (~500K tokens/mes), el coste de API sigue siendo
   bajo (< 100 euros/mes con modelos eficientes). El punto de inflexión
   hacia fine-tuning suele estar en decenas de millones de tokens/mes.

## 7. Recapitulación: el modelo es un commodity, el contexto es la ventaja

| Pregunta | Respuesta |
|---|---|
| ¿Por que ya no hay que entrenar para tener capacidad de IA avanzada? | Los modelos fundacionales accesibles por API ofrecen capacidad superior a lo que la mayoria de empresas podrian entrenar |
| ¿Cual es la ventaja competitiva real en un mundo API-first? | Datos propietarios, proceso integrado y velocidad de iteración |
| ¿Cuando entrenar es obligatorio? | Regulación de dato, latencia crítica, confidencialidad irrenunciable, volumen que hace la API mas cara que infra propia |
| ¿Cuando entrenar es un error? | Cuando la tarea ya la resuelve un modelo fundacional con buen prompting y el tiempo de go-to-market es crítico |
| ¿Que hace que un proyecto de IA basado en API sea defendible? | Integración profunda de dato propio (RAG, fine-tuning) que la competencia no puede replicar con la misma API |

## 8. Assets generados

| Fichero | Contenido |
|---|---|
| `images/B05E_fig01.png` | Línea de tiempo: tres épocas de acceso a IA (2012-2024) |
| `images/B05E_fig02.png` | Comparativa costes log-scale + perfil estratégico por enfoque |
| `images/B05E_fig03.png` | Diagrama: fuentes de ventaja en empresa API-first |
| `images/B05E_fig04.png` | Árbol de decisión make vs buy |
| `images/B05E_fig05.png` | TCO comparativo 12 meses con desglose por componente |

**Dependencias:** numpy, pandas, matplotlib